In [ ]:
import ee
import json
from datetime import datetime

try:
    from ee_ipl_uv import multitemporal_cloud_masking
    from ee_ipl_uv import image_wrapper
    print("Import ee_ipl_uv succues!")
except ImportError as e:
    print(f"Libary not found. {e}")

project_id = 'civic-athlete-496122-c4'
try:
    ee.Initialize(project=project_id)
    print("GEE initilized!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=project_id)

Import von ee_ipl_uv erfolgreich!
GEE erfolgreich initialisiert!


In [ ]:
geojson_pfad = "Landkreis_Wuerzburg.geojson" 

with open(geojson_pfad, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

if geojson_data.get("type") == "FeatureCollection":
    region_of_interest = ee.FeatureCollection(geojson_data).geometry()
else:
    region_of_interest = ee.Geometry(geojson_data)

print("succues")

GeoJSON erfolgreich geladen und in GEE-Geometrie umgewandelt!


In [ ]:
start_date = '2025-04-01'
end_date = '2025-06-30'

landsat_collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
                      .filterDate(start_date, end_date)
                      .filterBounds(region_of_interest))

landsat_ids = landsat_collection.aggregate_array('system:index').getInfo()
print(f"Number of Landsat photos: {len(landsat_ids)}")
if len(landsat_ids) > 0:
    print(f"Example Landsat-IDs: {landsat_ids[:3]}")

sentinel_collection = (ee.ImageCollection('COPERNICUS/S2')
                       .filterDate(start_date, end_date)
                       .filterBounds(region_of_interest))

sentinel_ids = sentinel_collection.aggregate_array('system:index').getInfo()
print(f"\n Number of Sentinel-2 photos: {len(sentinel_ids)}")
if len(sentinel_ids) > 0:
    print(f"Example Sentinel-IDs: {sentinel_ids[:3]}")

Gefundene Landsat Bilder im Zeitraum: 21
Beispiel Landsat-IDs: ['LC08_194025_20250412', 'LC08_194025_20250428', 'LC08_194025_20250514']


c:\Users\Linus\miniconda3\envs\ee\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for COPERNICUS/S2! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by COPERNICUS/S2_HARMONIZED

Learn more: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2

  warnings.warn(warning, category=DeprecationWarning)



Gefundene Sentinel-2 Bilder im Zeitraum: 109
Beispiel Sentinel-IDs: ['20250403T102559_20250403T103147_T32UNA', '20250403T102559_20250403T103147_T32UNV', '20250405T102041_20250405T102051_T32UNA']


In [ ]:
landsat_info = landsat_collection.reduceColumns(
    ee.Reducer.toList().repeat(2), ['system:index', 'CLOUD_COVER']
).getInfo()

landsat_ids_list = landsat_info['list'][0]
cloud_percentages = landsat_info['list'][1]

print(f"{'Index':<6} | {'Landsat ID':<25} | {'offical clouds':<20}")
print("-" * 60)
for i in range(len(landsat_ids_list)):
    print(f"{i:<6} | {landsat_ids_list[i]:<25} | {cloud_percentages[i]:.2f}%")

print("-" * 70)

for i, img_id in enumerate(landsat_ids_list):
    img_temp = ee.Image(f'LANDSAT/LC08/C02/T1_TOA/{img_id}')
    
    url = img_temp.getThumbURL({
        'bands': ['B4', 'B3', 'B2'],
        'min': 0.0,
        'max': 0.3,
        'dimensions': 512, 
        'format': 'png',
        'region': region_of_interest
    })
    print(f"Index [{i}] -> {img_id} ({cloud_percentages[i]:.1f}% clouds):")
    print(f"{url}\n")

Index  | Landsat Bild-ID           | Offizielle Bewölkung
------------------------------------------------------------
0      | LC08_194025_20250412      | 1.85%
1      | LC08_194025_20250428      | 2.42%
2      | LC08_194025_20250514      | 0.35%
3      | LC08_194025_20250530      | 48.74%
4      | LC08_194025_20250615      | 70.92%
5      | LC08_194026_20250412      | 0.04%
6      | LC08_194026_20250428      | 1.36%
7      | LC08_194026_20250514      | 1.89%
8      | LC08_194026_20250530      | 25.74%
9      | LC08_194026_20250615      | 87.85%
10     | LC08_195025_20250403      | 0.21%
11     | LC08_195025_20250419      | 56.18%
12     | LC08_195025_20250521      | 77.78%
13     | LC08_195025_20250606      | 85.45%
14     | LC08_195025_20250622      | 1.00%
15     | LC08_195026_20250403      | 0.13%
16     | LC08_195026_20250419      | 14.44%
17     | LC08_195026_20250505      | 86.45%
18     | LC08_195026_20250521      | 86.91%
19     | LC08_195026_20250606      | 93.44%
20     | L

In [ ]:
if len(landsat_ids) > 0:
    test_id_landsat = landsat_ids[8]
    print(f"Landsat-picture: {test_id_landsat}")
    
    img_wrapper_l8 = image_wrapper.L8L1TImage(test_id_landsat, collection='LANDSAT/LC08/C02/T1_TOA')
    
    cloud_mask_l8, forecast_img_l8 = multitemporal_cloud_masking.CloudClusterScore(
        img_wrapper_l8, 
        region_of_interest, 
        method_pred="persistence"
    )
    
    mask_url_l8 = cloud_mask_l8.getThumbURL({
        'min': 0, 
        'max': 2, 
        'palette': 'blue,white,red',
        'dimensions': 1024,
        'format': 'png',
        'region': region_of_interest
    })
    print(f"Landsat cloud-mask Link: {mask_url_l8}")
else:
    print("no picture")

Verarbeite Landsat-Bild: LC08_194026_20250530
Landsat Wolkenmaske Vorschau-Link: https://earthengine.googleapis.com/v1/projects/civic-athlete-496122-c4/thumbnails/9467e2d36450f2759ee2c333412cf199-52cbbfb2f473057fcf16813fe0e4e224:getPixels


In [ ]:
original_img = img_wrapper_l8.ee_img

vorher_url = original_img.getThumbURL({
    'bands': ['B4', 'B3', 'B2'],
    'min': 0.0,
    'max': 0.3,
    'dimensions': 1024,
    'format': 'png',
    'region': region_of_interest
})
print(f"1. Before (with clouds): \n{vorher_url}\n")

masked_img = original_img.updateMask(cloud_mask_l8.eq(0))

nachher_maskiert_url = masked_img.getThumbURL({
    'bands': ['B4', 'B3', 'B2'],
    'min': 0.0,
    'max': 0.3,
    'dimensions': 1024,
    'format': 'png',
    'region': region_of_interest
})
print(f"2. After (found clouds): \n{nachher_maskiert_url}\n")

nachher_forecast_url = forecast_img_l8.getThumbURL({
    'bands': ['B4_forecast', 'B3_forecast', 'B2_forecast'],
    'min': 0.0,
    'max': 0.3,
    'dimensions': 1024,
    'format': 'png',
    'region': region_of_interest
})
print(f"3. After (ground truth of model): \n{nachher_forecast_url}\n")

1. VORHER (Original-Bild mit Wolken): 
https://earthengine.googleapis.com/v1/projects/civic-athlete-496122-c4/thumbnails/4262d79c116ac45033a4dec8d1c9f18a-fe9e64de2bea32efeaca7258d216a50a:getPixels

2. NACHHER (Option A - Wolken herausgeschnitten): 
https://earthengine.googleapis.com/v1/projects/civic-athlete-496122-c4/thumbnails/af078adfd1097fa57a9543fba3727274-59fdc599cad1205045240c9d11e920fa:getPixels

3. NACHHER (Option B - Vom Modell geschätzter wolkenfreier Hintergrund): 
https://earthengine.googleapis.com/v1/projects/civic-athlete-496122-c4/thumbnails/1e0ef8f19b315fc2f9009caa5bb9a9ac-b9d750c04986bb4dc29c084809982f2a:getPixels



In [ ]:
sentinel_info = sentinel_collection.reduceColumns(
    ee.Reducer.toList().repeat(2), ['system:index', 'CLOUDY_PIXEL_PERCENTAGE']
).getInfo()

sentinel_ids_list = sentinel_info['list'][0]
s2_cloud_percentages = sentinel_info['list'][1]

print(f"{'Index':<6} | {'Sentinel-2 ID':<55} | {'offical clouds':<20}")
print("-" * 90)
for i in range(len(sentinel_ids_list)):
    print(f"{i:<6} | {sentinel_ids_list[i]:<55} | {s2_cloud_percentages[i]:.2f}%")

print("\n" + "="*80)
print("="*80)

for i, img_id in enumerate(sentinel_ids_list):
    img_temp = ee.Image(f'COPERNICUS/S2_HARMONIZED/{img_id}')
    
    url = img_temp.getThumbURL({
        'bands': ['B4', 'B3', 'B2'],
        'min': 0,
        'max': 3000,
        'dimensions': 512,
        'format': 'png',
        'region': region_of_interest
    })
    print(f"Index [{i}] -> {img_id} ({s2_cloud_percentages[i]:.1f}% Wolken):")
    print(f"{url}\n")
    if i == 30:
        break

Index  | Sentinel-2 Bild-ID                                      | Offizielle Bewölkung
------------------------------------------------------------------------------------------
0      | 20250403T102559_20250403T103147_T32UNA                  | 0.00%
1      | 20250403T102559_20250403T103147_T32UNV                  | 1.00%
2      | 20250405T102041_20250405T102051_T32UNA                  | 8.33%
3      | 20250405T102041_20250405T102051_T32UNV                  | 1.45%
4      | 20250407T101701_20250407T101810_T32UNA                  | 0.00%
5      | 20250407T101701_20250407T101810_T32UNV                  | 0.00%
6      | 20250408T103051_20250408T103158_T32UNA                  | 6.50%
7      | 20250408T103051_20250408T103158_T32UNV                  | 16.34%
8      | 20250410T101559_20250410T101701_T32UNA                  | 75.04%
9      | 20250410T101559_20250410T101701_T32UNV                  | 23.20%
10     | 20250410T102701_20250410T103346_T32UNA                  | 70.09%
11     | 20250

KeyboardInterrupt: 

In [ ]:
if len(sentinel_ids) > 0:
    test_id_s2 = sentinel_ids[2]
    print(f"Verarbeite Sentinel-Bild: {test_id_s2}")
    
    img_wrapper_s2 = image_wrapper.S2L1CImage(test_id_s2, collection='COPERNICUS/S2_HARMONIZED')
    
    cloud_mask_s2, forecast_img_s2 = multitemporal_cloud_masking.CloudClusterScore(
        img_wrapper_s2, 
        region_of_interest, 
        method_pred="persistence"
    )
    
    mask_url_s2 = cloud_mask_s2.getThumbURL({
        'min': 0, 
        'max': 2, 
        'palette': 'blue,white,red',
        'dimensions': 1024,
        'format': 'png',
        'region': region_of_interest
    })
    print(f"Sentinel-2 cloud-mask Link: {mask_url_s2}")
else:
    print("no picture")

Verarbeite Sentinel-Bild: 20250405T102041_20250405T102051_T32UNA
Sentinel-2 Wolkenmaske Vorschau-Link: https://earthengine.googleapis.com/v1/projects/civic-athlete-496122-c4/thumbnails/d6aa0ea36dae102c8c0d3d0affe4553f-c61123fc8a796027a3a3291543921847:getPixels


In [ ]:
original_img_s2 = img_wrapper_s2.ee_img

vorher_url_s2 = original_img_s2.getThumbURL({
    'bands': ['B4', 'B3', 'B2'],
    'min': 0.0,
    'max': 0.3,
    'dimensions': 1024,
    'format': 'png',
    'region': region_of_interest
})
print(f"1. Before (Original-Picture with clouds): \n{vorher_url_s2}\n")

masked_img_s2 = original_img_s2.updateMask(cloud_mask_s2.eq(0))

nachher_maskiert_url_s2 = masked_img_s2.getThumbURL({
    'bands': ['B4', 'B3', 'B2'],
    'min': 0.0,
    'max': 0.3,
    'dimensions': 1024,
    'format': 'png',
    'region': region_of_interest
})
print(f"2. After (found clouds): \n{nachher_maskiert_url_s2}\n")

nachher_forecast_url_s2 = forecast_img_s2.getThumbURL({
    'bands': ['B4_forecast', 'B3_forecast', 'B2_forecast'],
    'min': 0.0,
    'max': 0.3,
    'dimensions': 1024,
    'format': 'png',
    'region': region_of_interest
})
print(f"3. After (ground truth of model): \n{nachher_forecast_url_s2}\n")

1. VORHER (Original-Bild mit Wolken): 
https://earthengine.googleapis.com/v1/projects/civic-athlete-496122-c4/thumbnails/aaeb71e5b8285cf39006d49d2bcf5328-ce7980c84797a449cc04ac3ade3d59c0:getPixels

2. NACHHER (Option A - Wolken herausgeschnitten): 
https://earthengine.googleapis.com/v1/projects/civic-athlete-496122-c4/thumbnails/bfef4ab321aeb41b8b5482081b4df7f8-0241afed5df01953e7ba4272ba1d03d8:getPixels

3. NACHHER (Option B - Vom Modell geschätzter wolkenfreier Hintergrund): 
https://earthengine.googleapis.com/v1/projects/civic-athlete-496122-c4/thumbnails/4bf457968abebcbdd495bbba8c23126e-c7f6adb434c7414453510d0703c97bea:getPixels

